In [ ]:
## Imports

import sys, os
from pathlib import Path

parent_folder = str(Path.cwd().parents[1])
if parent_folder not in sys.path:
    sys.path.append(parent_folder)

from sigpy import mri
import sigpy.plot as pl
import scipy
import pickle
import sigpy as sp
import cupy as cp
import numpy as np
from scipy.io import savemat, loadmat
import twixtools
import matplotlib.pyplot as plt


## My files
import save_data_helpers
import recon_plot_helpers
from math import ceil
from custom_recons.created_app import TotalVariationRecon_Stacked, _stacked_nufft_operator_sens
from custom_recons.tv_app import TotalVariation_Phase_Stacked
from coils import coil_utils

In [ ]:
data_bins = save_data_helpers.read_pickle('/data/lilianae/data_and_spoke_bins/patient3_mid0283_data_bins_5gates.pkl')
dcf_bins = save_data_helpers.read_pickle('/data/lilianae/subject3_mid0283_processed/dcf_512_5gates.pkl')
spoke_bins = save_data_helpers.read_pickle('/data/lilianae/data_and_spoke_bins/patient3_mid0283_spoke_bins_5gates.pkl')
mps = np.ones((15,58,512,512), dtype=np.complex128)

In [ ]:
num_gates = len(data_bins)
data_bins_with_dcf = [None] * num_gates

## Create list of data_bins with dcf applied
for gate in range(num_gates):
    data_bins_with_dcf[gate] = data_bins[gate] * dcf_bins[gate]
    # print(f'Data bins w/ dcf shape = {data_bins_with_dcf[gate].shape}')
    # print(f'coords shape = {spoke_bins[gate].shape}')


In [ ]:
def nufft_apod_1d(n, oversamp=1.25, width=4):
    beta = np.pi * (((width / oversamp) * (oversamp - 0.5))**2 - 0.8)**0.5
    os_n = ceil(oversamp*n)
    idx = np.arange(n, dtype=np.float64)

    apod = (beta**2 - (np.pi * width * (idx - n//2) / os_n)**2)**0.5
    apod /= np.sinh(apod)
    return apod.astype(np.float32)

In [ ]:
# def stacked_nufft_sens(
#         img_shape: tuple,
#         coords,
#         mps, oversamp=1.25, width=4):
#     """setup a stacked 2D NUFFT sp operator acting on a 3D image
#        the opeator first performs a 1D FFT along the "z" axis (0 or left-most axis)
#        followed by applying 2D NUFFTS to all "slices"
       
#     Parameters
#     ----------
#         img_shape: tuple
#             shape of the image
#         coords: (numpy or cupy) array 
#             coordinates of the k-space samples
#             shape (n_k_space_points,2)
#             units: "unitless" -> -N/2 ... N/2 at Nyquist (sp convention)
#         mps: (numpy or cupy) array
#             sensitivity maps of shape (num_channels, *img_shape)

#     Returns
#     -------
#         Diag: a stack of NUFFT operators
#     """

#     num_channels = len(mps)
#     num_slices, nx, ny = img_shape

#     apod_z = nufft_apod_1d(num_slices)
#     apod_z = apod_z.reshape(-1, 1, 1)   # (num_slices, 1, 1)
#     print(f'apod_z.shape = {apod_z.shape}')


#     os_num_slices = ceil(oversamp * num_slices)
#     z_scale = os_num_slices / (num_slices ** 0.5)

#     # ------------------------------------------------------------------
#     # 1. FFT along z
#     # ------------------------------------------------------------------
#     ft_z = sp.linop.FFT(img_shape, axes=(0,))

#     # setup a 2D NUFFT operator for the start
#     nufft_op = sp.linop.NUFFT(img_shape[1:], coords)


#     # reshaping operator for input
#     rs_in = sp.linop.Reshape(img_shape[1:], (1, ) + img_shape[1:])
#     # setup a list of "n" 2D NUFFT operators (one per slice)
#     ops = []
#     for i in range(img_shape[0]):
#         coords_i = coords[i].reshape(-1, coords.shape[-1])[:, 1:]  # (400*512, 2)
#         nufft_op_i = sp.linop.NUFFT(img_shape[1:], coords_i)
#         # Reshape NUFFT output from flat to 2D: (400*512,) -> (400, 512)
#         rs_nufft = sp.linop.Reshape((coords.shape[1], coords.shape[2]), nufft_op_i.oshape)
#         rs_out_i = sp.linop.Reshape((1, coords.shape[1], coords.shape[2]), (coords.shape[1], coords.shape[2]))
#         if i==0:
#             print(f'nufft_op_i = {nufft_op_i}')
#             print(f'rs_nufft = {rs_nufft}')
#             print(f'rs_out_i = {rs_out_i}')
#         ops.append(rs_out_i * rs_nufft * nufft_op_i * rs_in)

#     # Do:
#     apod_z_correction = 1.0 / (apod_z * z_scale)  # values ~1.4 to ~8.2 -> BRIGHTENS

#     apod_z_gpu = cp.array(
#         np.broadcast_to(apod_z_correction, img_shape).copy(),
#         dtype=cp.complex64
#     )
#     ApodZ = sp.linop.Multiply(img_shape, apod_z_gpu)

#     # apply 2D NUFFTs to all "slices" using the sp Diag operator
#     full_op= sp.linop.Diag(ops, iaxis=0, oaxis=0) * ft_z
#     #### Combine Sensitivity Op (mult with sens) and respective ft0+nuFFT op:

#     #sensitivity = np.ones((num_channels,*img_shape),dtype=np.complex64)
#     S = sp.linop.Multiply(img_shape,mps)

#     rs_in_sense = sp.linop.Reshape(img_shape,(1,)+img_shape)
#     rs_out_sense = sp.linop.Reshape((1,)+tuple(full_op.oshape),full_op.oshape)
#     return  sp.linop.Diag(num_channels*[rs_out_sense*full_op*rs_in_sense],iaxis=0,oaxis=0)*S, apod_z, z_scale



In [ ]:
spoke_bins[0].shape

In [ ]:
print(spoke_bins[0][:, 100, 250, 0])

In [ ]:
coords = spoke_bins[0]
# Verify z-coords are identical across spokes and readouts within each slice
z_all = coords[..., 0]   # (58, 368, 512)
for i in range(58):
    assert np.allclose(z_all[i], z_all[i, 0, 0]), f"Slice {i} z-coords not uniform"
print("z-coords uniform per slice: OK")
print(f"z values: {z_all[:, 0, 0]}")  # should be 58 distinct uniform values

In [ ]:
# def stacked_nufft_sens(img_shape, coords, mps):
#     """setup a stacked 2D NUFFT sp operator acting on a 3D image
#        the opeator first performs a 1D FFT along the "z" axis (0 or left-most axis)
#        followed by applying 2D NUFFTS to all "slices"
       
#     Parameters
#     ----------
#         img_shape: tuple
#             shape of the image
#         coords: (numpy or cupy) array 
#             coordinates of the k-space samples
#             shape (n_k_space_points,2)
#             units: "unitless" -> -N/2 ... N/2 at Nyquist (sp convention)
#         mps: (numpy or cupy) array
#             sensitivity maps of shape (num_channels, *img_shape)

#     Returns
#     -------
#         Diag: a stack of NUFFT operators
#     """

#     num_channels = len(mps)

#     z_coords = coords[:, 0, 0, 0:1].reshape(-1)
#     ft0_op = sp.linop.NUFFT((img_shape[0],1,1), coord=z_coords)

#     # setup a 2D NUFFT operator for the start
#     nufft_op = sp.linop.NUFFT(img_shape[1:], coords)


#     # reshaping operator for input
#     rs_in = sp.linop.Reshape(img_shape[1:], (1, ) + img_shape[1:])
#     # setup a list of "n" 2D NUFFT operators (one per slice)
#     ops = []
#     for i in range(img_shape[0]):
#         coords_i = coords[i].reshape(-1, coords.shape[-1])[:, 1:]  # (400*512, 2)
#         nufft_op_i = sp.linop.NUFFT(img_shape[1:], coords_i)
#         # Reshape NUFFT output from flat to 2D: (400*512,) -> (400, 512)
#         rs_nufft = sp.linop.Reshape((coords.shape[1], coords.shape[2]), nufft_op_i.oshape)
#         rs_out_i = sp.linop.Reshape((1, coords.shape[1], coords.shape[2]), (coords.shape[1], coords.shape[2]))
#         ops.append(rs_out_i * rs_nufft * nufft_op_i * rs_in)


#     # apply 2D NUFFTs to all "slices" using the sp Diag operator
#     full_op= sp.linop.Diag(ops, iaxis=0, oaxis=0) * ft0_op
#     #### Combine Sensitivity Op (mult with sens) and respective ft0+nuFFT op:

#     #sensitivity = np.ones((num_channels,*img_shape),dtype=np.complex64)
#     S = sp.linop.Multiply(img_shape,mps)

#     rs_in_sense = sp.linop.Reshape(img_shape,(1,)+img_shape)
#     rs_out_sense = sp.linop.Reshape((1,)+tuple(full_op.oshape),full_op.oshape)
#     return  sp.linop.Diag(num_channels*[rs_out_sense*full_op*rs_in_sense],iaxis=0,oaxis=0)*S

In [ ]:
def stacked_nufft_sens(img_shape, coords, mps):
    num_channels = mps.shape[0]
    num_slices, nx, ny = img_shape
    num_spokes   = coords.shape[1]
    num_readouts = coords.shape[2]

    # z coords: one unique value per slice, shape (num_slices, 1) for ndim=1 NUFFT
    z_coords = coords[:, 0, 0, 0:1]  # (num_slices, 1)

    # Per-slice 2D xy coords
    coords_2d = [
        coords[i, :, :, 1:].reshape(-1, 2)  # (spokes*readouts, 2)
        for i in range(num_slices)
    ]

    class StackedNUFFT(sp.linop.Linop):
        def __init__(self):
            self.nufft_2d = [
                sp.linop.NUFFT((nx, ny), coords_2d[i])
                for i in range(num_slices)
            ]
            # Batched z-NUFFT:
            #   ishape = (nx*ny, num_slices)  -- nx*ny independent 1D signals of length num_slices
            #   coord  = (num_slices, 1)      -- ndim=1, so acts on last axis (num_slices)
            #   oshape = (nx*ny, num_slices)  -- num_slices output points per pixel
            self.nufft_z = sp.linop.NUFFT((nx * ny, num_slices), z_coords)

            super().__init__(
                oshape=[num_slices, num_spokes, num_readouts],
                ishape=list(img_shape)
            )

        def _apply(self, input):
            # input: (num_slices, nx, ny)
            xp = sp.get_array_module(input)

            # Step 1: z-NUFFT (batched over all nx*ny pixels)
            # (num_slices, nx, ny) -> (nx*ny, num_slices) -> nufft_z -> (nx*ny, num_slices)
            #                      -> (num_slices, nx, ny)
            img_flat = input.reshape(num_slices, nx * ny).T   # (nx*ny, num_slices)
            z_flat   = self.nufft_z(img_flat)                 # (nx*ny, num_slices)
            z_img    = z_flat.T.reshape(num_slices, nx, ny)   # (num_slices, nx, ny)

            # Step 2: 2D NUFFT per slice
            output = xp.zeros([num_slices, num_spokes, num_readouts], dtype=input.dtype)
            for i in range(num_slices):
                output[i] = self.nufft_2d[i](z_img[i]).reshape(num_spokes, num_readouts)
            return output

        def _adjoint_linop(self):
            return StackedNUFFTAdjoint(self.nufft_2d, self.nufft_z)

    class StackedNUFFTAdjoint(sp.linop.Linop):
        def __init__(self, nufft_2d, nufft_z):
            self.nufft_2d = nufft_2d
            self.nufft_z  = nufft_z  # same batched op, .H gives adjoint z-NUFFT
            super().__init__(
                oshape=list(img_shape),
                ishape=[num_slices, num_spokes, num_readouts]
            )

        def _apply(self, input):
            # input: (num_slices, num_spokes, num_readouts)
            xp = sp.get_array_module(input)

            # Step 1: adjoint 2D NUFFT per slice (handles xy apodization fully)
            xy_img = xp.zeros(img_shape, dtype=input.dtype)
            for i in range(num_slices):
                ksp_flat   = input[i].reshape(-1)          # (spokes*readouts,)
                xy_img[i]  = self.nufft_2d[i].H(ksp_flat) # (nx, ny)

            # Step 2: adjoint 1D z-NUFFT (batched, handles z apodization fully)
            # (num_slices, nx, ny) -> (nx*ny, num_slices) -> nufft_z.H -> (nx*ny, num_slices)
            #                      -> (num_slices, nx, ny)
            xy_flat = xy_img.reshape(num_slices, nx * ny).T  # (nx*ny, num_slices)
            out_flat = self.nufft_z.H(xy_flat)               # (nx*ny, num_slices)
            output   = out_flat.T.reshape(num_slices, nx, ny) # (num_slices, nx, ny)

            return output

        def _adjoint_linop(self):
            return StackedNUFFT()

    # ── Assemble ──
    stacked_op = StackedNUFFT()

    print(f'stacked_op  : {stacked_op}')   # (58,512,512) -> (58,368,512)
    print(f'nufft_z     : {stacked_op.nufft_z}')  # (262144,58) -> (262144,58)

    # ── Sensitivity maps ──
    S           = sp.linop.Multiply(img_shape, mps)
    rs_in_coil  = sp.linop.Reshape(img_shape, (1,) + img_shape)
    rs_out_coil = sp.linop.Reshape((1,) + tuple(stacked_op.oshape),
                                    tuple(stacked_op.oshape))
    per_coil_op = rs_out_coil * stacked_op * rs_in_coil

    A = sp.linop.Diag(
        num_channels * [per_coil_op], iaxis=0, oaxis=0
    ) * S

    print(f'A (full)    : {A}')
    return A

In [ ]:
gate = 0
device = 2

with cp.cuda.Device(device):
    ksp_gpu    = sp.to_device(data_bins_with_dcf[gate], device)
    coords_gpu = sp.to_device(spoke_bins[gate], device)
    mps_gpu    = sp.to_device(mps, device)

    # # ── Reference: sp.nufft_adjoint, single coil, no sens ──
    # ref = sp.nufft_adjoint(
    #     data_bins_with_dcf[gate][0],                           # single coil: (58, 368, 512)
    #     coord=spoke_bins[gate]          # (58, 368, 512, 3)
    # )
    # print(f'ref shape:  {ref.shape}')
    # print(f'ref max:    {np.abs(ref).max():.6e}')
    # print(f'ref mean:   {np.abs(ref).mean():.6e}')

    # ── Stacked: single coil, no sens, NO apod/scale yet ──
    A_bare= stacked_nufft_sens(
        img_shape=(58, 512, 512),
        coords=coords_gpu,
        mps=mps_gpu
    )
    result_bare = A_bare.H(ksp_gpu)
    result_bare = sp.to_device(result_bare, -1)

    # Take coil 0 only for comparison
    result_bare_c0 = result_bare[0]
    print(f'bare shape: {result_bare_c0.shape}')
    print(f'bare max:   {np.abs(result_bare_c0).max():.6e}')
    print(f'bare mean:  {np.abs(result_bare_c0).mean():.6e}')

    # ── Ratio: what scale factor separates them? ──
    # ratio = np.abs(ref).mean() / (np.abs(result_bare_c0).mean() + 1e-20)
    # print(f'scale ratio (ref/bare): {ratio:.6e}')

    # # ── Check apod_z values ──
    # print(f'apod_z min/max: {apod_z.min():.6e} / {apod_z.max():.6e}')
    # print(f'z_scale: {z_scale:.6e}')
    # print(f'apod_z * z_scale min/max: {(apod_z*z_scale).min():.6e} / {(apod_z*z_scale).max():.6e}')

In [ ]:
# Check your spoke_bins coords:
print(f'coord z  range: {spoke_bins[0][...,0].min():.3f} to {spoke_bins[0][...,0].max():.3f}')
print(f'coord x  range: {spoke_bins[0][...,1].min():.3f} to {spoke_bins[0][...,1].max():.3f}')
print(f'coord y  range: {spoke_bins[0][...,2].min():.3f} to {spoke_bins[0][...,2].max():.3f}')
# They must span -256 to +256 for a 512-sized dim, and -29 to +29 for 58 slices
# If not, rescale: coords_fixed = coords * (N/2) / coords.max()

In [ ]:
gate0_recon = recon_plot_helpers.crop_xy_dimension(cp.asnumpy(result_bare), oshape=(58, 256, 256))
recon_plot_helpers.plot_recons_all_axes(gate0_recon, z_idx=34, title=rf"Stacked NUFFT Operator")